In [ ]:
!pip install flashlight-text

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import glob
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import transforms
from torchaudio.models.decoder import ctc_decoder

In [ ]:
"""
  dataset class for the images, constructed from csv files containing pixel values of the preprocessed images
"""
class WordSet(Dataset):
  def __init__(self, state):
    dset = np.random.random((1, 128*32))
    self.labels = []
    if state == "train":
      path = "drive/MyDrive/word_detect/files/file*"
    else:
      path = "drive/MyDrive/word_detect/files/test/file*"
    for filename in glob.glob(path):
      df = pd.read_csv(filename)
      dset = np.vstack((dset, np.array(df.iloc[:, 1:-1], dtype = "float32")))
      self.labels += df.iloc[:, -1].tolist()
    dset = dset[1:]
    self.tensor = torch.from_numpy(dset).reshape((len(dset), 1, 32, 128)).to(dtype = torch.float32)
    mean, std = self.tensor.mean(), self.tensor.std()
    transform = transforms.Normalize((mean,), (std,))
    self.tensor = transform(self.tensor)


  def __len__(self):
    return self.tensor.shape[0]
  def __getitem__(self, ndx):
    return self.tensor[ndx], self.labels[ndx]

In [ ]:
class CRNN(nn.Module):
  def  __init__(self, charno):
    super(CRNN, self).__init__()
    self.charno = charno
    self.model1 = nn.Sequential(nn.Conv2d(1, 32, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2,2)),

                          nn.Conv2d(32, 64, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2,2)),
                          nn.Dropout2d(0.25),

                          nn.Conv2d(64, 128, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2, 1)),
                          nn.Dropout2d(0.25),

                          nn.Conv2d(128, 256, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2, 1)),
                          nn.Dropout2d(0.25),

                          nn.Conv2d(256, 256, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2, 1)),
                          nn.Dropout2d(0.25)).to(dtype = torch.float32)
    self.model2 = nn.Sequential(nn.Linear(256, 32),
                                nn.LSTM(32, 128, bidirectional=True, batch_first=True)).to(dtype = torch.float32)
    self.model3 = nn.LSTM(256, 128, bidirectional=True, batch_first=True).to(dtype = torch.float32)
    self.model4 = nn.Linear(256, self.charno)
  def forward(self, x):
    out = self.model1(x)
    batch_size, channels, height, width = out.size()
    out = out.permute(0,2,1,3).view(batch_size, width, -1)
    out, _ = self.model2(out)
    out, _ = self.model3(out)
    out = self.model4(out)
    return out


In [ ]:
characters = " ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
char_to_idx = {char: idx for idx, char in enumerate(characters)}
lex = dict(enumerate(characters))

def encode_label(label, char_to_idx):
  return [char_to_idx[char] for char in label]

def decode(labels):
  return "".join([characters[i] for i in labels if i != 0])

def CTC_args(labels, batch_size):
  label_arr = [encode_label(label, char_to_idx) + [0]*(32-len(label)) for label in labels]
  labels_tensor = torch.tensor(label_arr)

  input_lengths = torch.tensor([32]*batch_size)
  label_lengths = torch.tensor([len(label) for label in labels])
  return labels_tensor, label_lengths, input_lengths

"""def decode_pred(logits):
  logits = logits.squeeze().T
  values = logits.T.max(dim = 0).indices
  result = [0]
  for i in values:
    if i.item() == result[-1]:
      continue
    else:
      result.append(i.item())
  return decode(result)"""


character_list = list(characters)
decoder = ctc_decoder(
    lexicon=None,
    tokens=character_list,
    lm=None,
    beam_size=10,
    blank_token=" ",
    sil_token = " ")

def decode_pred(log_probs):
  results = decoder(log_probs, torch.tensor([53]))
  tokens = results[0][0].tokens.tolist()
  return decode(tokens)


In [ ]:
# Define the key components
model = CRNN(len(characters))  # CRNN model
criterion = nn.CTCLoss(blank=0, zero_infinity=True)  # Assuming blank label is 0
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
train_batch, test_batch = 128, 10
train = WordSet("train")
test = WordSet("test")
trainloader = DataLoader(train, train_batch, shuffle=True, drop_last=True)
testloader = DataLoader(test, test_batch, shuffle=True, drop_last=True)

In [ ]:

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    #model.train()
    epoch_loss = 0

    for batch_idx, (images, labels) in enumerate(trainloader):


        # Forward pass
        logits = model(images)  # Shape: (batch_size, 32, num_classes)
        log_probs = nn.functional.log_softmax(logits, dim=2)  # Apply log softmax
        log_probs = log_probs.permute(1, 0, 2)  # (seq_len, batch_size, num_classes)

        labels_tensor, label_lengths, input_lengths = CTC_args(labels, train_batch)

        # Compute input lengths (all are 32 in this case)

        #print(log_probs.shape, labels_tensor.shape, label_lengths.shape, input_lengths.shape)
        # Compute CTC Loss
        loss = criterion(log_probs, labels_tensor, input_lengths, label_lengths)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track loss
        epoch_loss += loss.item()

        if batch_idx % 40 == 0:
            print(f"Epoch [{epoch}/{num_epochs}], Batch [{batch_idx+1}], Loss: {loss.item():.4f}")

    test_loss = 0

    for _, (test_images, test_labels) in enumerate(testloader):
      with torch.no_grad():
        logits = model(test_images)  # Shape: (batch_size, 32, num_classes)
        log_probs = nn.functional.log_softmax(logits, dim=2)  # Apply log softmax
        log_probs = log_probs.permute(1, 0, 2)  # (seq_len, batch_size, num_classes)

        test_labels_tensor, test_label_lengths, test_input_lengths = CTC_args(test_labels, test_batch)

        test_loss += criterion(log_probs, test_labels_tensor, test_input_lengths, test_label_lengths)

    print(f"Epoch [{epoch}/{num_epochs}] Average Loss: {epoch_loss / len(trainloader):.4f}")
    print(f"\tAverage Loss: {test_loss / len(testloader):.4f}")

    if epoch%2 == 0:
      print("Sample Predictions: \n")
      for log_prob, label in zip(log_probs.permute(1,0,2), test_labels):
        print("Pred: {},   True: {}".format(decode_pred(log_prob.unsqueeze(0)), label))

Epoch [0/10], Batch [1], Loss: 37.7553
Epoch [0/10], Batch [41], Loss: 3.5563
Epoch [0/10], Batch [81], Loss: 3.4138
Epoch [0/10] Average Loss: 5.9022
	Average Loss: 3.4060
Sample Predictions: 

Pred: ,   True: As
Pred: ,   True: which
Pred: ,   True: The
Pred: ,   True: God
Pred: ,   True: but
Pred: ,   True: see
Pred: ,   True: new
Pred: ,   True: by
Pred: ,   True: which
Pred: RWeQYoNcHydoTeJUqVgL,   True: still
Epoch [1/10], Batch [1], Loss: 3.3646
Epoch [1/10], Batch [41], Loss: 3.2695
Epoch [1/10], Batch [81], Loss: 3.2411
Epoch [1/10] Average Loss: 3.3347
	Average Loss: 3.2626
Epoch [2/10], Batch [1], Loss: 3.2553
Epoch [2/10], Batch [41], Loss: 3.2163
Epoch [2/10], Batch [81], Loss: 3.2381


KeyboardInterrupt: 